# RL-token-minimize — full pipeline on Kaggle

Runs the project end to end on a Kaggle GPU: data prep → baseline eval → LoRA SFT → post-SFT eval → smoke RL.

**Setup**: Settings → Accelerator → GPU (T4 or P100). Internet must be enabled (Settings → Internet → On) to clone the repo and download the model/dataset. On T4 ×2 only the first GPU is used — HF Trainer's DataParallel wrapping breaks with the PEFT setup, so the setup cell pins `CUDA_VISIBLE_DEVICES=0`.

Artifacts land in `/kaggle/working/RL-token-minimize/checkpoints/` — download them from the Output tab.

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
%cd /kaggle/working
!rm -rf RL-token-minimize
!git clone --depth 1 https://github.com/augustoFranke/RL-token-minimize.git
%cd RL-token-minimize
!mkdir -p checkpoints
!pip uninstall -q -y torchao
!pip install -q -U "transformers>=5.13" "trl>=1.7.1" "peft>=0.19.1" "datasets>=5.0.0" "accelerate>=1.14.0" pytest
!pip install -q -e . --no-deps
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| bf16", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

In [ ]:
!python -m pytest -q

In [ ]:
!python scripts/prepare_data.py
!wc -l data/*.jsonl

In [ ]:
!python scripts/evaluate.py --limit 5 --out checkpoints/eval_base5.jsonl

In [ ]:
!python scripts/train_sft.py

In [ ]:
!python scripts/evaluate.py --adapter checkpoints/sft --limit 5 --out checkpoints/eval_sft5.jsonl

In [ ]:
!python scripts/train_rl.py --no-thinking --steps 10 --output checkpoints/rl_smoke
!tail -3 checkpoints/rl_smoke/log.jsonl

## Full runs (uncomment once the smoke run looks healthy)

```
!python scripts/evaluate.py --adapter checkpoints/sft --out checkpoints/eval_sft_full.jsonl
!python scripts/train_rl.py --no-thinking --output checkpoints/rl_run1
!python scripts/train_rl.py --thinking --token-cost all --output checkpoints/rl_run2
!python scripts/train_rl.py --thinking --token-cost final --output checkpoints/rl_run3
!python scripts/evaluate.py --adapter checkpoints/rl_run1/final --out checkpoints/eval_rl1.jsonl
```